# Step 10 — Model Comparison: Clustering & Soft Probabilities

This notebook compares multiple clustering methods (KMeans, GMM, HMM, Spectral)
on the same PCA-reduced feature space. Two sections:

**Part A — Hard Clustering Comparison:**
Side-by-side PCA scatters, Adjusted Rand Index (ARI) agreement matrix,
temporal label agreement, and stacked regime timelines.

**Part B — Soft Probability Comparison:**
GMM and HMM posterior probability stacked area charts,
sharpness comparison, and Markov 2-state recession overlay.

**Run `python run_pipeline.py --steps 3` before executing this notebook.**

## Setup & Load Data (D8a.1)

In [ ]:
%matplotlib inline
import sys
sys.path.insert(0, "../src")
import logging
import subprocess
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from trading_crab_lib.config import load, setup_logging
from trading_crab_lib.runtime import RunConfig
from trading_crab_lib.checkpoints import CheckpointManager
from trading_crab_lib import DATA_DIR, OUTPUT_DIR, plotting
from trading_crab_lib.clustering import reduce_pca

setup_logging("INFO")
log = logging.getLogger("10_model_comparison")
cfg = load()
run_cfg = RunConfig(generate_plots=True, save_plots=True, show_plots=False)
cm = CheckpointManager()

In [ ]:
def run_step_if_needed(step: int, required_paths: list, auto_run: bool = True) -> bool:
    """Run the pipeline step if any required output files are missing."""
    missing = [p for p in required_paths if not Path(p).exists()]
    if not missing:
        return True
    print(f"Missing: {[str(p) for p in missing]}")
    scripts = sorted(Path("../pipelines").glob(f"{step:02d}_*.py"))
    if not scripts:
        print(f"No pipeline script found for step {step}.")
        return False
    script = scripts[0]
    if not auto_run:
        print(f"  \u2192 Run: python {script}")
        return False
    print(f"  \u2192 Running {script.name} ...")
    result = subprocess.run(["python", str(script)], capture_output=True, text=True, cwd="..")
    out = result.stdout
    if len(out) > 4000:
        out = out[:2000] + "\n...\n" + out[-2000:]
    print(out)
    if result.returncode != 0:
        print("STDERR:", result.stderr[-1000:])
        return False
    print(f"  \u2713 Step {step} complete.")
    return True

In [ ]:
# Load features and compute PCA
features = None
pca_df = None
pca_obj = None
kmeans_labels = None
regime_names = {}

try:
    features = cm.load("features")
    print(f"Features loaded: {features.shape}")
except Exception as e:
    print(f"Features not found: {e}")
    print("Run: python run_pipeline.py --steps 2")

if features is not None:
    clust_cols = cfg.get("features", {}).get("clustering_features", [])
    avail = [c for c in clust_cols if c in features.columns]
    feat_narrow = features[avail].dropna()
    n_comp = cfg.get("clustering", {}).get("n_pca_components", 5)
    pca_df, pca_obj, pca_scaler = reduce_pca(feat_narrow, n_components=n_comp)
    print(f"PCA: {pca_df.shape} ({n_comp} components)")

# Load KMeans labels
try:
    cluster_labels = cm.load("cluster_labels")
    if "balanced_cluster" in cluster_labels.columns:
        kmeans_labels = cluster_labels["balanced_cluster"]
    elif "cluster" in cluster_labels.columns:
        kmeans_labels = cluster_labels["cluster"]
    print(f"KMeans labels loaded: {kmeans_labels.nunique()} regimes, {len(kmeans_labels)} quarters")
except Exception:
    print("Cluster labels not found — run step 3.")

# Load regime names
try:
    import yaml
    names_path = Path("../config/regime_labels.yaml")
    if names_path.exists():
        regime_names = yaml.safe_load(names_path.read_text()) or {}
        regime_names = {int(k): v for k, v in regime_names.items()}
except Exception:
    pass
if not regime_names and kmeans_labels is not None:
    regime_names = {i: f"Regime {i}" for i in sorted(kmeans_labels.dropna().astype(int).unique())}
print(f"Regime names: {regime_names}")

In [ ]:
# Fit alternative clustering methods (GMM, HMM, Spectral)
labels_dict = {}
gmm_model = None
hmm_model = None

if pca_df is not None and kmeans_labels is not None:
    common = pca_df.index.intersection(kmeans_labels.dropna().index)
    labels_dict["KMeans"] = kmeans_labels.loc[common].astype(int)
    k = labels_dict["KMeans"].nunique()
    pca_common = pca_df.loc[common]

    # GMM
    try:
        from trading_crab_lib.gmm import fit_gmm, select_gmm_k, gmm_labels
        bic_df, gmm_models, gmm_scaler = fit_gmm(pca_common, k_range=range(2, k + 3))
        best_k, best_cov = select_gmm_k(bic_df)
        gmm_model = gmm_models[(best_k, best_cov)]
        labels_dict["GMM"] = gmm_labels(pca_common, gmm_model, gmm_scaler)
        print(f"GMM: k={best_k}, cov={best_cov}")
    except Exception as e:
        print(f"GMM fitting failed: {e}")

    # HMM (optional — requires hmmlearn)
    try:
        from trading_crab_lib.hmm import fit_hmm, select_hmm_k, hmm_labels
        hmm_scores, hmm_models, hmm_scaler = fit_hmm(pca_common, k_range=range(2, k + 3))
        best_hmm_k = select_hmm_k(hmm_scores)
        hmm_model = hmm_models[best_hmm_k]
        labels_dict["HMM"] = hmm_labels(pca_common, hmm_model, hmm_scaler)
        print(f"HMM: k={best_hmm_k}")
    except ImportError:
        print("hmmlearn not installed — skipping HMM. pip install hmmlearn")
    except Exception as e:
        print(f"HMM fitting failed: {e}")

    # Spectral
    try:
        from trading_crab_lib.spectral import spectral_labels
        labels_dict["Spectral"] = spectral_labels(pca_common, k=k)
        print(f"Spectral: k={k}")
    except Exception as e:
        print(f"Spectral fitting failed: {e}")

    print(f"\nMethods fitted: {list(labels_dict.keys())}")
else:
    print("PCA data or KMeans labels not available — cannot fit methods.")

## Side-by-Side PCA Scatter: 4 Methods (D8a.2)

PC1 vs PC2 scatter colored by cluster assignment from each method.
Same color scale across panels — note that cluster IDs may not align
across methods (e.g., KMeans regime 0 may correspond to GMM regime 3).

In [ ]:
if pca_df is not None and labels_dict:
    methods = list(labels_dict.keys())
    n_methods = len(methods)
    fig, axes = plt.subplots(1, n_methods, figsize=(5 * n_methods, 5), squeeze=False)

    pc1, pc2 = pca_df.columns[0], pca_df.columns[1]
    common = pca_df.index
    for lbl in labels_dict.values():
        common = common.intersection(lbl.dropna().index)
    pca_plot = pca_df.loc[common]

    for idx, method in enumerate(methods):
        ax = axes[0][idx]
        lbl = labels_dict[method].loc[common].astype(int)
        for cid in sorted(lbl.unique()):
            mask = lbl == cid
            ax.scatter(
                pca_plot.loc[mask, pc1], pca_plot.loc[mask, pc2],
                c=plotting._regime_color(cid), s=20, alpha=0.7,
                label=f"C{cid}", edgecolors="none"
            )
        ax.set_xlabel(pc1)
        ax.set_ylabel(pc2 if idx == 0 else "")
        ax.set_title(method, fontsize=11)
        ax.legend(fontsize=7, loc="upper right")
        ax.grid(alpha=0.2)

    fig.suptitle("PCA Scatter — Clustering Method Comparison", fontsize=13, y=1.02)
    fig.tight_layout()
    plotting._save_or_show(fig, "10_pca_scatter_comparison.png", run_cfg)
else:
    print("No clustering results available.")

## ARI Pairwise Matrix Heatmap (D8a.3)

Adjusted Rand Index (ARI) between every pair of methods.
ARI = 1.0 means identical clustering; ARI ~ 0 means random agreement.
High ARI between KMeans and GMM suggests robust cluster structure.

In [ ]:
if len(labels_dict) >= 2:
    from trading_crab_lib.cluster_comparison import pairwise_rand_index
    import seaborn as sns

    ari_df = pairwise_rand_index(labels_dict)
    print("Pairwise Adjusted Rand Index:")
    display(ari_df.round(3))

    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        ari_df, annot=True, fmt=".3f", cmap="YlOrRd", vmin=0, vmax=1,
        square=True, linewidths=0.5, ax=ax
    )
    ax.set_title("Adjusted Rand Index — Pairwise Method Agreement", fontsize=11)
    fig.tight_layout()
    plotting._save_or_show(fig, "10_ari_heatmap.png", run_cfg)
else:
    print("Need at least 2 methods for ARI comparison.")

## Temporal Label Agreement (D8a.4)

For each quarter, check whether all methods agree on the same label.
Shows the fraction of quarters with full agreement, pairwise agreement,
and a time-series of agreement over the sample period.

In [ ]:
if len(labels_dict) >= 2:
    methods = list(labels_dict.keys())
    common = labels_dict[methods[0]].dropna().index
    for m in methods[1:]:
        common = common.intersection(labels_dict[m].dropna().index)

    # Build label matrix (rows=quarters, cols=methods)
    label_matrix = pd.DataFrame(
        {m: labels_dict[m].loc[common].astype(int) for m in methods},
        index=common
    )

    # Pairwise agreement fraction
    print("Pairwise label agreement (fraction of quarters):")
    for i, m1 in enumerate(methods):
        for m2 in methods[i+1:]:
            # Since cluster IDs may not match, use ARI-based agreement
            # For raw label match we'd need Hungarian alignment
            from sklearn.metrics import adjusted_rand_score
            ari = adjusted_rand_score(label_matrix[m1], label_matrix[m2])
            print(f"  {m1} vs {m2}: ARI = {ari:.3f}")

    # Rolling agreement: for each quarter, count how many method pairs agree
    # Use a simple metric: number of unique labels assigned by all methods
    n_unique = label_matrix.apply(lambda row: row.nunique(), axis=1)
    full_agree = (n_unique == 1).mean()
    print(f"\nFull agreement (all methods same label): {full_agree:.1%} of quarters")
    print(f"Note: raw label matching ignores ID permutation — use ARI for robust comparison.")

    # Plot: rolling window of unique-label count
    fig, ax = plt.subplots(figsize=(14, 3.5))
    rolling_unique = n_unique.rolling(8, min_periods=1).mean()
    ax.plot(rolling_unique.index, rolling_unique.values, color=plotting.CUSTOM_COLORS[0], linewidth=1.5)
    ax.axhline(1.0, color="green", linewidth=0.8, linestyle="--", alpha=0.5, label="Perfect agreement")
    ax.set_ylabel("Avg unique labels (8Q rolling)")
    ax.set_xlabel("Date")
    ax.set_title(f"Temporal Label Diversity — {len(methods)} Methods", fontsize=11)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)
    fig.tight_layout()
    plotting._save_or_show(fig, "10_temporal_agreement.png", run_cfg)
else:
    print("Need at least 2 methods for agreement analysis.")

## Regime Timeline Comparison (D8a.5)

Four stacked horizontal timelines — one per clustering method.
Each row shows which regime was active at each quarter.
Visually compare where methods agree and where they diverge.

In [ ]:
if labels_dict:
    methods = list(labels_dict.keys())
    n_methods = len(methods)
    fig, axes = plt.subplots(n_methods, 1, figsize=(16, 2 * n_methods), sharex=True, squeeze=False)

    for idx, method in enumerate(methods):
        ax = axes[idx][0]
        lbl = labels_dict[method].dropna().astype(int)
        unique_ids = sorted(lbl.unique())

        for dt, cid in lbl.items():
            ax.barh(0, width=92, left=dt, height=0.8,
                    color=plotting._regime_color(cid), alpha=0.85)

        ax.set_yticks([0])
        ax.set_yticklabels([method], fontsize=10, fontweight="bold")
        ax.set_ylim(-0.5, 0.5)

        # Legend for this method's clusters
        from matplotlib.patches import Patch
        patches = [Patch(color=plotting._regime_color(c), label=f"C{c}") for c in unique_ids]
        ax.legend(handles=patches, fontsize=7, loc="upper right", ncol=len(unique_ids))
        ax.grid(axis="x", alpha=0.2)

    axes[-1][0].set_xlabel("Date")
    fig.suptitle("Regime Timeline — Method Comparison", fontsize=13, y=1.01)
    fig.tight_layout()
    plotting._save_or_show(fig, "10_regime_timeline_comparison.png", run_cfg)
else:
    print("No clustering results available.")

---

## Part B — Soft Probabilities

GMM and HMM produce posterior probabilities for each quarter — a "soft"
assignment rather than a single hard label. Stacked area charts show how
confident each model is and where uncertainty concentrates.

## GMM Soft Probabilities (D8b.1)

In [ ]:
gmm_probs = None
if pca_df is not None and gmm_model is not None:
    from trading_crab_lib.gmm import gmm_probabilities
    common = pca_df.index.intersection(labels_dict.get("KMeans", pd.Series(dtype=int)).dropna().index)
    pca_common = pca_df.loc[common]
    gmm_probs = gmm_probabilities(pca_common, gmm_model, gmm_scaler)
    print(f"GMM probabilities: {gmm_probs.shape}")
    print(f"Mean max probability: {gmm_probs.max(axis=1).mean():.3f} (1.0 = perfectly sharp)")

    plotting.plot_soft_probabilities(
        gmm_probs, regime_names, run_cfg,
        title="GMM Soft Regime Probabilities",
        filename="10_gmm_soft_probabilities.png",
    )
else:
    print("GMM model not available — skipping.")

## HMM Soft Probabilities (D8b.2)

In [ ]:
hmm_probs = None
if pca_df is not None and hmm_model is not None:
    from trading_crab_lib.hmm import hmm_probabilities
    common = pca_df.index.intersection(labels_dict.get("KMeans", pd.Series(dtype=int)).dropna().index)
    pca_common = pca_df.loc[common]
    hmm_probs = hmm_probabilities(pca_common, hmm_model, hmm_scaler)
    print(f"HMM probabilities: {hmm_probs.shape}")
    print(f"Mean max probability: {hmm_probs.max(axis=1).mean():.3f} (1.0 = perfectly sharp)")

    plotting.plot_soft_probabilities(
        hmm_probs, regime_names, run_cfg,
        title="HMM Soft Regime Probabilities",
        filename="10_hmm_soft_probabilities.png",
    )
else:
    print("HMM model not available — skipping. pip install hmmlearn")

## GMM vs HMM Sharpness Comparison (D8b.3)

Compare how "sharp" (confident) each method's assignments are.
Higher entropy = more uncertain. HMM typically produces sharper
assignments because it models temporal transitions.

In [ ]:
if gmm_probs is not None or hmm_probs is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4), squeeze=False)

    def _entropy(probs_df):
        """Per-row Shannon entropy (nats)."""
        p = probs_df.values.clip(1e-12)
        return -(p * np.log(p)).sum(axis=1)

    models_to_plot = []
    if gmm_probs is not None:
        models_to_plot.append(("GMM", gmm_probs, plotting.CUSTOM_COLORS[0]))
    if hmm_probs is not None:
        models_to_plot.append(("HMM", hmm_probs, plotting.CUSTOM_COLORS[1]))

    # Panel 1: Entropy time-series
    ax1 = axes[0][0]
    for name, probs, color in models_to_plot:
        ent = _entropy(probs)
        ax1.plot(probs.index, ent, label=f"{name} (mean={np.mean(ent):.3f})",
                 color=color, linewidth=1, alpha=0.8)
    ax1.set_ylabel("Shannon Entropy (nats)")
    ax1.set_xlabel("Date")
    ax1.set_title("Assignment Uncertainty Over Time", fontsize=11)
    ax1.legend(fontsize=8)
    ax1.grid(alpha=0.2)

    # Panel 2: Max probability distribution
    ax2 = axes[0][1]
    for name, probs, color in models_to_plot:
        max_p = probs.max(axis=1)
        ax2.hist(max_p.values, bins=30, color=color, alpha=0.5,
                 label=f"{name} (median={max_p.median():.3f})", edgecolor="white")
    ax2.set_xlabel("Max Probability")
    ax2.set_ylabel("Count")
    ax2.set_title("Assignment Sharpness Distribution", fontsize=11)
    ax2.legend(fontsize=8)
    ax2.grid(alpha=0.2)

    fig.suptitle("GMM vs HMM — Soft Assignment Comparison", fontsize=13, y=1.02)
    fig.tight_layout()
    plotting._save_or_show(fig, "10_sharpness_comparison.png", run_cfg)

    # Summary table
    rows = []
    for name, probs, _ in models_to_plot:
        max_p = probs.max(axis=1)
        ent = _entropy(probs)
        rows.append({
            "Method": name,
            "Mean Max Prob": f"{max_p.mean():.3f}",
            "Median Max Prob": f"{max_p.median():.3f}",
            "Mean Entropy": f"{np.mean(ent):.3f}",
            "% Confident (>0.8)": f"{(max_p > 0.8).mean():.1%}",
        })
    display(pd.DataFrame(rows).set_index("Method"))
else:
    print("Neither GMM nor HMM probabilities available.")

## Markov 2-State Recession Overlay (D8b.4)

Fit a 2-state Markov regime-switching model on a key macro series
(e.g., GDP growth) and overlay its recession probability on the
KMeans regime timeline. Helps answer: which KMeans regimes are recessions?

In [ ]:
if features is not None and kmeans_labels is not None:
    try:
        from trading_crab_lib.markov import (
            fit_markov_switching, markov_labels, markov_probabilities,
            compare_markov_kmeans,
        )

        # Find a suitable series for Markov switching
        candidates = ["log_sp500_d1", "log_us_real_gdp_per_capita_d1", "log_us_cpi_d1"]
        series = None
        series_name = None
        for c in candidates:
            if c in features.columns:
                s = features[c].dropna()
                if len(s) > 40:
                    series = s
                    series_name = c
                    break

        if series is not None:
            print(f"Fitting 2-state Markov on: {series_name} ({len(series)} obs)")
            markov_result = fit_markov_switching(series, k_regimes=2)
            m_labels = markov_labels(markov_result)
            m_probs = markov_probabilities(markov_result)

            # Cross-tabulation
            crosstab = compare_markov_kmeans(markov_result, kmeans_labels)
            print("\nMarkov vs KMeans cross-tabulation:")
            display(crosstab)

            # Plot: KMeans timeline with Markov recession probability overlay
            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 5), sharex=True,
                                           gridspec_kw={"height_ratios": [1, 2]})

            # Top: Markov recession probability
            # Identify which Markov state is "recession" (lower mean of the series)
            state_means = series.groupby(m_labels.reindex(series.index)).mean()
            recession_state = state_means.idxmin()
            recession_col = m_probs.columns[recession_state] if recession_state < len(m_probs.columns) else m_probs.columns[0]
            recession_prob = m_probs[recession_col]

            ax1.fill_between(recession_prob.index, 0, recession_prob.values,
                             color=plotting.CUSTOM_COLORS[1], alpha=0.4)
            ax1.plot(recession_prob.index, recession_prob.values,
                     color=plotting.CUSTOM_COLORS[1], linewidth=0.8)
            ax1.set_ylabel("P(recession)")
            ax1.set_ylim(0, 1)
            ax1.set_title(f"Markov 2-State Recession Probability ({series_name})", fontsize=11)
            ax1.grid(alpha=0.2)

            # Bottom: KMeans regime timeline
            lbl = kmeans_labels.dropna().astype(int)
            for dt, cid in lbl.items():
                ax2.barh(0, width=92, left=dt, height=0.8,
                         color=plotting._regime_color(cid), alpha=0.85)
            ax2.set_yticks([0])
            ax2.set_yticklabels(["KMeans"], fontsize=10)
            ax2.set_ylim(-0.5, 0.5)
            ax2.set_xlabel("Date")
            from matplotlib.patches import Patch
            patches = [Patch(color=plotting._regime_color(c),
                             label=regime_names.get(c, f"R{c}"))
                       for c in sorted(lbl.unique())]
            ax2.legend(handles=patches, fontsize=7, loc="upper right",
                       ncol=len(patches))
            ax2.grid(axis="x", alpha=0.2)

            fig.suptitle("Markov Recession Probability vs KMeans Regimes", fontsize=13, y=1.01)
            fig.tight_layout()
            plotting._save_or_show(fig, "10_markov_recession_overlay.png", run_cfg)
        else:
            print("No suitable series found for Markov switching.")

    except ImportError:
        print("statsmodels not installed — skipping Markov. pip install statsmodels")
    except Exception as e:
        print(f"Markov fitting failed: {e}")
else:
    print("Features or KMeans labels not available.")